In [21]:
import pandas as pd
from pathlib import Path
from loader import Loader
from borough_identifier import BoroughIdentifier

In [11]:
DONNEES_URL = "https://www.donneesquebec.ca/recherche/api/3/action/"
SQL_URL = "datastore_search_sql?"
URL_REQUESTS = "dbfc05f8-b939-4639-ae52-2e77f738e43f"

output_path = Path.cwd().resolve().parent / "data" / "raw_requests.csv"
loader = Loader()

### Data checks

In [17]:
crime_sql = f"""sql=SELECT "ID_UNIQUE", "NATURE", "ARRONDISSEMENT", "ARRONDISSEMENT_GEO", "DDS_DATE_CREATION", "LOC_LONG", "LOC_LAT" from "{URL_REQUESTS}" where ("ARRONDISSEMENT" is not null or "ARRONDISSEMENT_GEO" is not null) and "NATURE" = 'Plainte' AND "DDS_DATE_CREATION" >= '2021-01-01' AND "DDS_DATE_CREATION" < '2022-01-01'"""
path = f"""{DONNEES_URL}{SQL_URL}{crime_sql}"""
path = path.replace(" ", "%20")
loader.load_to_csv(path, output_path)

In [18]:
df = pd.read_csv(output_path)
df.head(5)

,ID_UNIQUE,NATURE,ARRONDISSEMENT,ARRONDISSEMENT_GEO,DDS_DATE_CREATION,LOC_LONG,LOC_LAT
0,21-322785,Plainte,Montréal-Nord,Montréal-Nord,2021-12-02T14:25:41,-73.647011,45.590246
1,21-322823,Plainte,Lachine,Lachine,2021-12-02T14:52:27,-73.681918,45.439813
2,21-322833,Plainte,Le Plateau-Mont-Royal,Le Plateau-Mont-Royal,2021-12-02T14:58:02,-73.582048,45.535310
3,21-322847,Plainte,Le Plateau-Mont-Royal,Le Plateau-Mont-Royal,2021-12-02T15:05:05,-73.570186,45.514078
4,21-322873,Plainte,Côte-des-Neiges - Notre-Dame-de-Grâce,Côte-des-Neiges-Notre-Dame-de-Grâce,2021-12-02T15:25:58,-73.633162,45.471155


In [19]:
print(df["DDS_DATE_CREATION"].min())
print(df["DDS_DATE_CREATION"].max())

2021-01-01T09:50:46
2021-12-31T14:59:06


In [31]:
borough_identifier = BoroughIdentifier()
df["BOROUGH"] = df.apply(
    lambda row: borough_identifier.match_WSG84_to_borough(row["LOC_LONG"], row["LOC_LAT"]), # Because custom function does not accept full Series
    axis = 1
) 

not_matching = df["ARRONDISSEMENT_GEO"] != df["BOROUGH"]
df_not_matching = df[not_matching]
print(len(df_not_matching))

not_matching = df["ARRONDISSEMENT"] != df["BOROUGH"]
df_not_matching = df[not_matching]
print(len(df_not_matching))

86
3589


In [33]:
df["ARRONDISSEMENT"].value_counts()

ARRONDISSEMENT
Le Plateau-Mont-Royal                         751
Ahuntsic - Cartierville                       670
Rosemont - La Petite-Patrie                   639
Villeray-Saint-Michel - Parc-Extension        510
Mercier - Hochelaga-Maisonneuve               471
Côte-des-Neiges - Notre-Dame-de-Grâce         449
Rivière-des-Prairies - Pointe-aux-Trembles    443
Ville-Marie                                   441
Montréal-Nord                                 275
Le Sud-Ouest                                  226
Pierrefonds - Roxboro                         222
LaSalle                                       217
Saint-Léonard                                 206
Verdun                                        189
Lachine                                       161
Anjou                                         133
L'Île-Bizard - Sainte-Geneviève               133
Saint-Laurent                                 109
Outremont                                     106
Ville de Montréal                  

In [32]:
df["ARRONDISSEMENT_GEO"].value_counts()

ARRONDISSEMENT_GEO
Le Plateau-Mont-Royal                       754
Ahuntsic-Cartierville                       668
Rosemont-La Petite-Patrie                   637
Villeray-Saint-Michel-Parc-Extension        512
Mercier-Hochelaga-Maisonneuve               473
Côte-des-Neiges-Notre-Dame-de-Grâce         446
Rivière-des-Prairies-Pointe-aux-Trembles    443
Ville-Marie                                 441
Montréal-Nord                               275
Le Sud-Ouest                                227
Pierrefonds-Roxboro                         222
LaSalle                                     218
Saint-Léonard                               206
Verdun                                      187
Lachine                                     161
L'Île-Bizard-Sainte-Geneviève               133
Anjou                                       132
Saint-Laurent                               109
Outremont                                   105
Côte-Saint-Luc                                1
Mont-Royal           

In [35]:
print(df["ARRONDISSEMENT"].isna().sum())
print(df["ARRONDISSEMENT_GEO"].isna().sum())

0
26


### Preprocessing

In [38]:
crime_sql = f"""sql=SELECT "ID_UNIQUE", "NATURE", "ARRONDISSEMENT", "ARRONDISSEMENT_GEO" from "{URL_REQUESTS}" where ("ARRONDISSEMENT" is not null or "ARRONDISSEMENT_GEO" is not null) and "NATURE" = 'Plainte' AND "DDS_DATE_CREATION" >= '2021-01-01' AND "DDS_DATE_CREATION" < '2022-01-01'"""
path = f"""{DONNEES_URL}{SQL_URL}{crime_sql}"""
path = path.replace(" ", "%20")
loader.load_to_csv(path, output_path)

In [39]:
df = pd.read_csv(output_path)

In [42]:
df["BOROUGH"] = df["ARRONDISSEMENT_GEO"].fillna(df["ARRONDISSEMENT"])
print(df["BOROUGH"].isna().sum())

0


In [43]:
df["BOROUGH"].value_counts()

BOROUGH
Le Plateau-Mont-Royal                       754
Ahuntsic-Cartierville                       668
Rosemont-La Petite-Patrie                   637
Villeray-Saint-Michel-Parc-Extension        512
Mercier-Hochelaga-Maisonneuve               473
Côte-des-Neiges-Notre-Dame-de-Grâce         446
Rivière-des-Prairies-Pointe-aux-Trembles    443
Ville-Marie                                 441
Montréal-Nord                               275
Le Sud-Ouest                                227
Pierrefonds-Roxboro                         222
LaSalle                                     218
Saint-Léonard                               206
Verdun                                      187
Lachine                                     161
L'Île-Bizard-Sainte-Geneviève               133
Anjou                                       132
Saint-Laurent                               109
Outremont                                   105
Ville de Montréal                            26
Côte-Saint-Luc                  

In [45]:
not_boroughs = ["Ville de Montréal", "Côte-Saint-Luc", "Mont-Royal", "Hampstead", "Westmount"]
df = df[~df["BOROUGH"].isin(not_boroughs)]

In [46]:
counts = df["BOROUGH"].value_counts().reset_index()
counts = counts.rename(columns={
    "count" : "complaints"
})
counts

,BOROUGH,complaints
0,Le Plateau-Mont-Royal,754
1,Ahuntsic-Cartierville,668
2,Rosemont-La Petite-Patrie,637
3,Villeray-Saint-Michel-Parc-Extension,512
4,Mercier-Hochelaga-Maisonneuve,473
5,Côte-des-Neiges-Notre-Dame-de-Grâce,446
6,Rivière-des-Prairies-Pointe-aux-Trembles,443
7,Ville-Marie,441
8,Montréal-Nord,275
9,Le Sud-Ouest,227


In [47]:
counts.to_csv("../data/requests_dataset.csv", index = False)